# Why tokenization matters 
Before writing a single line of code, let's understand why tokenization is one of the most important (and most misunderstood) parts of a language model. Almost every "weird" LLM behavior traces back here.

### Some problems that tokenization can cause 
1. LLM can't spell words correctly 
2. LLM struggles with string reversal 
3. LLM is worse at non-English text 
4. LLM fails at simple arithmatic 
5. GPT-2 was bad at Python Indentation 
6. <|endoftext> causes abrupt halt 
7. YAML preferred over JSON for LLMs 

Tokenizer is a completely separate module from the LLM itself. It has its own training data 
and its own training process. Once trained, it translates between raw text and sequences of 
integer token IDs. The LLM only sees integers, never the raw text. 

![Tokenization](./images/workflow.png)

**Note**: this is inpsired from Karpathy's video

# What is a Token 
A token is the smallest unit of text that a model processes. Tokens are not characters, words, or syllables — they're chunks learned from data.

Experiment: Visualize Tokenization
Visit https://tiktokenizer.vercel.app and paste this text:

```text
The café serves crème brûlée daily.
مرحبا بالعالم
def add(a, b):
    return a + b
```

Observe how:

- English words often become single tokens
- Arabic text fragments into many small tokens
- Code with consistent indentation patterns merges well
- The same word with different capitalization gets different tokens

This visual intuition is essential before diving into code.


## Characters, Unicode, and Bytes 
### Unicode 

Every character in every language has a code point — a unique number assigned by the Unicode standard. Currently, Unicode supports over 1.1 million code points.

In [1]:
text = "café ☕ مرحبا"

# ord() gives the Unicode code point of a character
code_points = [ord(ch) for ch in text]
print("Text:        ", text)
print("Code points: ", code_points)
print("Length:      ", len(text))

Text:         café ☕ مرحبا
Code points:  [99, 97, 102, 233, 32, 9749, 32, 1605, 1585, 1581, 1576, 1575]
Length:       12


Notice: é (U+00E9) has code point 233. The coffee cup ☕ is 9749. Arabic letters are in the 1500s range.

Python give us three representations of any string. 

In [2]:
sample = "café ☕"

# 1: Raw characters
print("Characters: ", list(sample))

# 2: Unicode code points (what Python stores internally)
print("Code points:", [ord(c) for c in sample])

# 3: UTF-8 bytes (what we'll use for BPE!)
print("UTF-8 bytes:", list(sample.encode("utf-8")))
print("Byte length:", len(sample.encode("utf-8")))

Characters:  ['c', 'a', 'f', 'é', ' ', '☕']
Code points: [99, 97, 102, 233, 32, 9749]
UTF-8 bytes: [99, 97, 102, 195, 169, 32, 226, 152, 149]
Byte length: 9


- Chars: code points treated as text units. seq of unicode points. 
- code points: every unicode char has a unique integer identity called a code point, written as U+XXXX in hex. 
- utf-8 bytes: utf-8 is a variable length encoding that represents each code point as 1 to 4 bytes. given that integer(unicode point) what specific sequence of bytes do i write to disk, network, memory to represent it. Integer to bytes.  


### UTF-8 encoding
Why UTF-8? UTF-8 is the encoding used everywhere on the web. It maps every Unicode code point to 1–4 bytes. For BPE, we start at the byte level — our initial vocabulary is all 256 possible byte values (0–255).

 
| Code point range | Bytes used | Binary pattern |
|---|---|---|
| U+0000 – U+007F(0-255) | 1 byte | `0xxxxxxx` |
| U+0080 – U+07FF | 2 bytes | `110xxxxx 10yyyyyy` |
| U+0800 – U+FFFF | 3 bytes | `1110xxxx 10yyyyyy 10zzzzzz` |
| U+10000 – U+10FFFF | 4 bytes | `11110xxx 10yyyyyy 10zzzzzz 10wwwwww` |

ASCII characters (a–z, A–Z, 0–9, basic punctuation) fit in 1 byte. Non-ASCII characters require more.


In [3]:
examples = {
    "ASCII 'a'": "a",
    "Accented 'é'": "é",
    "Arabic 'م'": "م",
    "Emoji '🔥'": "🔥",
}

for label, char in examples.items():
    raw = char.encode("utf-8")
    print(f"{label}: {list(raw)}  ({len(raw)} bytes)")

ASCII 'a': [97]  (1 bytes)
Accented 'é': [195, 169]  (2 bytes)
Arabic 'م': [217, 133]  (2 bytes)
Emoji '🔥': [240, 159, 148, 165]  (4 bytes)


This is why LLMs are worse at non-English text: Arabic or Chinese text requires many more tokens than equivalent English text, eating more of the context window.

## The BPE Algorithm
 
Byte Pair Encoding (BPE) was originally a data compression algorithm. For tokenization, it works as follows:
 
### The Core Idea
 
1. Start with bytes as your initial vocabulary (256 tokens)
2. Find the most frequently occurring *pair* of consecutive tokens in your training text
3. Merge that pair into a new single token
4. Repeat until you reach your desired vocabulary size


### Example
 
Suppose your training corpus (after encoding to bytes) is:
```
[104, 101, 108, 108, 111, 32, 104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100]
```
(This is "hello hello world" in ASCII bytes. As ASCII is the the Byte representation)
 
**Round 1:** The pair `(108, 108)` ("ll") appears twice. Merge it into token `256`.
```
[104, 101, 256, 111, 32, 104, 101, 256, 111, 32, 119, 111, 114, 256, 100]
```
(Length drops from 17 → 15)
 
**Round 2:** Now `(104, 101)` ("he") appears twice. Merge into token `257`.
```
[257, 256, 111, 32, 257, 256, 111, 32, 119, 111, 114, 256, 100]
```
(Length drops from 15 → 13)
 
Each round compresses the text more. After enough rounds, common words, subwords, and even phrases become single tokens.


In [5]:

training_text = """
The night sky has fascinated humans for millennia. Ancient civilizations mapped 
the stars and used them for navigation, agriculture, and religious ceremonies. 
Modern astronomy has revealed that the universe contains hundreds of billions 
of galaxies, each harboring hundreds of billions of stars. The Milky Way, our 
home galaxy, spans roughly 100,000 light-years across. At its center lies a 
supermassive black hole called Sagittarius A*, with a mass of about 4 million 
suns. Beyond our galaxy, the nearest large neighbor is the Andromeda Galaxy, 
approximately 2.537 million light-years away. These vast distances are measured 
using techniques like parallax, Cepheid variables, and Type Ia supernovae. 
The expansion of the universe was discovered in 1929 by Edwin Hubble, who 
observed that distant galaxies are receding from us at speeds proportional to 
their distance — a relationship now called Hubble's Law. This expansion implies 
that the universe began in an extremely hot, dense state approximately 13.8 
billion years ago — the Big Bang. Cosmic microwave background radiation, 
discovered in 1965, provides a snapshot of the early universe when it was just 
380,000 years old. Dark matter, which does not emit or absorb light, makes up 
about 27% of the universe's total energy content, while dark energy accounts 
for approximately 68%, driving the accelerating expansion we observe today.
Astronomers use telescopes operating across the electromagnetic spectrum: 
radio waves, infrared, visible light, ultraviolet, X-rays, and gamma rays. 
Space missions like Hubble Space Telescope, James Webb Space Telescope, 
Chandra X-ray Observatory, and Fermi Gamma-ray Space Telescope have 
transformed our understanding of cosmic phenomena ranging from exoplanet 
atmospheres to the mergers of neutron stars.
"""

# 1: Encode to UTF-8 bytes
tokens = training_text.encode("utf-8")

# 2: Convert to a list of integers (0–255)
tokens = list(tokens)

print(f"Text length (characters): {len(training_text)}")
print(f"Token length (bytes):     {len(tokens)}")
print(f"First 40 bytes: {tokens[:40]}")

Text length (characters): 1822
Token length (bytes):     1826
First 40 bytes: [10, 84, 104, 101, 32, 110, 105, 103, 104, 116, 32, 115, 107, 121, 32, 104, 97, 115, 32, 102, 97, 115, 99, 105, 110, 97, 116, 101, 100, 32, 104, 117, 109, 97, 110, 115, 32, 102, 111, 114]


In [8]:
# Count all consecutive pairs

def get_stats(ids):
    """
    Given a list of integers, return a dictionary of 
    {(int, int): count} for every consecutive pair.
    
    Example:
        ids = [1, 2, 3, 2, 3]
        returns: {(1,2): 1, (2,3): 2, (3,2): 1}
    """
    counts = {}
    # ids=[1,2,3] → zip gives (1,2), (2,3)
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts



stats = get_stats(tokens)

# Show the top 10 most common pairs
top_pairs = sorted(stats.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 most common byte pairs:")
for pair, count in top_pairs:
    # Decode pair to see what characters they represent
    try:
        chars = bytes(pair).decode("utf-8", errors="replace")
    except:
        chars = "???"
        
    print(f"  Pair {pair} = {repr(chars):12s}  → count: {count}")

Top 10 most common byte pairs:
  Pair (101, 32) = 'e '          → count: 45
  Pair (115, 32) = 's '          → count: 43
  Pair (44, 32) = ', '          → count: 26
  Pair (101, 114) = 'er'          → count: 26
  Pair (32, 97) = ' a'          → count: 25
  Pair (104, 101) = 'he'          → count: 22
  Pair (32, 10) = ' \n'         → count: 22
  Pair (97, 110) = 'an'          → count: 21
  Pair (111, 110) = 'on'          → count: 21
  Pair (101, 115) = 'es'          → count: 21


In [10]:
# Merge the repeated pairs

def merge(ids, pair, idx):
    """
    Go through the token list, and whenever you see this pair (a, b), 
    replace it with a new token idx.
    
    Args:
        ids:  list of integers (current token sequence)
        pair: tuple (a, b) — the pair to merge
        idx:  integer — the new token ID to replace the pair with
    
    Returns:
        new list of integers with all occurrences of pair replaced
    
    Example:
        merge([1, 2, 3, 1, 2], (1, 2), 99)
        → [99, 3, 99]
    """
    new_ids = []
    i = 0
    
    while i < len(ids):
        # Check if we're NOT at the last position AND the pair matches
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i += 2  # skip both elements of the pair
        else:
            new_ids.append(ids[i]) # append the same element
            i += 1
    return new_ids



# test the function
test_ids = [5, 6, 6, 7, 5, 6, 8, 5]
print("Before merge:", test_ids)
result = merge(test_ids, (5, 6), 99)
print("After  merge:", result)
print()

# Now do the first real merge on our training text
top_pair = max(stats, key=stats.get)
print(f"Most frequent pair: {top_pair}")
print(f"Frequency: {stats[top_pair]}")

tokens2 = merge(tokens, top_pair, 256)
print(f"Length before merge: {len(tokens)}")
print(f"Length after  merge: {len(tokens2)}")

Before merge: [5, 6, 6, 7, 5, 6, 8, 5]
After  merge: [99, 6, 7, 99, 8, 5]

Most frequent pair: (101, 32)
Frequency: 45
Length before merge: 1826
Length after  merge: 1781


### The full BPE training loop

count all pairs -> pick max -> merge everywhere -> repeat vocab_size - 256 times. 

In [21]:
# How large do we want our vocabulary?
# Start: 256 (all possible bytes)
# End:   vocab_size
# Number of merges = vocab_size - 256

vocab_size = 300    # GPT-4 uses 100,277
num_merges = vocab_size - 256

ids = list(tokens)  
# make a copy, don't modify the original

merges = {}  # stores: (int, int) → int  (pair → new token id)

print(f"Starting BPE training: {num_merges} merges to perform")
print(f"Starting token sequence length: {len(ids)}")
print()

for i in range(num_merges):
    # count all current pairs
    stats = get_stats(ids)
    
    # find the most frequent pair
    pair = max(stats, key=stats.get)
    
    # assign a new token ID (starts at 256)
    idx = 256 + i
    
    # perform the merge
    ids = merge(ids, pair, idx)
    
    # record this merge rule for later use in encoding
    merges[pair] = idx
    
    print(f"  Merge {i+1:3d}: {pair} → {idx}  (freq={stats[pair]}, seq_len={len(ids)})")

print()
print(f"Final token sequence length: {len(ids)}")

Starting BPE training: 44 merges to perform
Starting token sequence length: 1826

  Merge   1: (101, 32) → 256  (freq=45, seq_len=1781)
  Merge   2: (115, 32) → 257  (freq=43, seq_len=1738)
  Merge   3: (44, 32) → 258  (freq=26, seq_len=1712)
  Merge   4: (101, 114) → 259  (freq=26, seq_len=1686)
  Merge   5: (97, 110) → 260  (freq=21, seq_len=1665)
  Merge   6: (111, 110) → 261  (freq=21, seq_len=1644)
  Merge   7: (100, 32) → 262  (freq=19, seq_len=1625)
  Merge   8: (97, 114) → 263  (freq=19, seq_len=1606)
  Merge   9: (116, 32) → 264  (freq=18, seq_len=1588)
  Merge  10: (116, 104) → 265  (freq=18, seq_len=1570)
  Merge  11: (105, 110) → 266  (freq=15, seq_len=1555)
  Merge  12: (108, 105) → 267  (freq=15, seq_len=1540)
  Merge  13: (121, 32) → 268  (freq=14, seq_len=1526)
  Merge  14: (97, 116) → 269  (freq=14, seq_len=1512)
  Merge  15: (114, 111) → 270  (freq=14, seq_len=1498)
  Merge  16: (108, 101) → 271  (freq=13, seq_len=1485)
  Merge  17: (265, 256) → 272  (freq=12, seq_len

Every iteration in this `for` loop does two full passes over `ids`: one inside `get_stats` and one inside `merge`(rebuils new_ids from scratch). 

Cost here: 
- cost = O(merges x corpus size)
- here = 44 x 1826 = 80,000 something operations 

Suppose you have vocab_size of `7936` which is still very small, and trained on say 10 million tokens then: 
- M x n = 7936 x 10,000,000 = 79 Billion. 

But this naive way of building tokenization is good for understanding. 

**Note: we will try to build the efficient version of tokenization in rest of the project outside this notebook**

In [15]:
# Measure how well we compressed
# compression ratio: how many original tokens fit into one compressed token (on average) 

original_length = len(tokens)
compressed_length = len(ids)
ratio = original_length / compressed_length

print(f"Original byte sequence length: {original_length}")
print(f"Compressed token sequence length: {compressed_length}")
print(f"Compression ratio: {ratio:.2f}x")
print(f"(The compressed version is {ratio:.2f}x shorter than the original)")

Original byte sequence length: 1826
Compressed token sequence length: 1241
Compression ratio: 1.47x
(The compressed version is 1.47x shorter than the original)


With only 44 merges on a small text, we already achieve 1.39x compression. Real models use 50,000–100,000 merges on gigabytes of text, achieving much higher ratios. This means a 4,096 token context window covers roughly 4,096 × compression_ratio characters of text.

#### Is Higher Compression Good for LLMs?
BPE is trying to balance two things: fewer tokens(faster and cheaper), and meaningful pieces(easier for the model to learn language).

Too much compression destroys the second one. 
Suppose tokenizer learns: 

`Un + break + able` 

This is GREAT because:
- reusable pieces
- model learns structure
- works for many words

At too much compression, tokenizer moves towards creating giant tokens like full words: `unbreakable`. But the sweet spot is subwords. 

In practice, modern LLM tokenizers usually aim for a balance where:
`compression ratio = 3x to 5x` means 3-5 raw bytes become roughly 1 token on average. Make it further simple. If your original UTF-8 byte stream is `1000 bytes`, after tokenization you may get `200-350 tokens`. 

### Build the vocabulary dictionary

In [16]:
# Start with the 256-base byte tokens
# Each maps to a single byte
vocab = {
    idx: bytes([idx]) for idx in range(256)
}

# Then add each merged token by concatenating its two component byte sequences
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
    # eg. vocab[256] = b'h' + b'e'

# Let's inspect some entries
print("Sample vocabulary entries:")
print()
for i in [65, 97, 200, 256, 257, 270]:
    if i in vocab:
        try:
            decoded = vocab[i].decode("utf-8", errors="replace")
        except:
            decoded = "<binary>"
        print(f"  Token {i:5d} → bytes {str(list(vocab[i])):20s} → text: {repr(decoded)}")

Sample vocabulary entries:

  Token    65 → bytes [65]                 → text: 'A'
  Token    97 → bytes [97]                 → text: 'a'
  Token   200 → bytes [200]                → text: '�'
  Token   256 → bytes [101, 32]            → text: 'e '
  Token   257 → bytes [115, 32]            → text: 's '
  Token   270 → bytes [114, 111]           → text: 'ro'


#### Decode 
Concatenate bytes for each id, decode UTF-8 once at the end, not per-token. 

In [17]:
# The decode function

def decode(ids):
    """
    Given a list of token IDs, return the corresponding text.
    
    Process:
    1. Look up each ID in vocab to get its bytes
    2. Concatenate all bytes together
    3. Decode the byte string as UTF-8
    
    Why errors='replace'?
    A sequence of token IDs might not form valid UTF-8 at every boundary.
    We use 'replace' to avoid crashes — invalid sequences become '?'.
    """
    # Concatenate byte strings for each token
    token_bytes = b"".join(vocab[idx] for idx in ids)
    
    # Decode the whole concatenated byte string as UTF-8
    text = token_bytes.decode("utf-8", errors="replace")
    return text

# Test decode
print(decode([84, 104, 101, 32]))  # T, h, e, space
print(decode([256]))               # Should be 'e '
print(decode([257]))               # Should be 'th'
print()

# Decode a sequence from our training
sample_ids = ids[:20]
print("First 20 token IDs from training:", sample_ids)
print("Decoded:", repr(decode(sample_ids)))

The 
e 
s 

First 20 token IDs from training: [10, 84, 104, 256, 110, 105, 291, 264, 115, 107, 268, 104, 97, 257, 102, 97, 115, 99, 266, 269]
Decoded: '\nThe night sky has fascinat'


#### Encode
apply merges in order they were learned (lowest merge-id first), not in the order pairs appear positionally. 
Not merge left to right.

In [ ]:

def encode(text):
    """
    Given a string, return a list of token IDs.
    
    Process:
    1. Convert text to UTF-8 bytes → list of integers (0–255)
    2. Repeatedly find the pair that should be merged next
       (the pair with the lowest-numbered merge — i.e., most frequent)
    3. Apply that merge
    4. Repeat until no more merges apply
    
    Key insight: we must apply merges in the SAME ORDER they were learned.
    We do this by picking the pair whose merge index is smallest.
    """
    tokens = list(text.encode("utf-8"))
    
    while len(tokens) >= 2:
        # Find all pairs in the current token list
        stats = get_stats(tokens)
        
        # the merge that was learned earliest in training gets applied first here..
        # merges.get(p, float("inf")) returns infinity for pairs not in merges,
        # so pairs that were never merged will never be chosen.
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        
        # If no pair in our current sequence is in merges, we're done
        if pair not in merges:
            break
        
        # Apply the merge
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    
    return tokens

# Test encode
print(encode("hello"))
print(encode("the universe"))
print()

# Round-trip test: encode then decode should give back the original
test_phrase = "The night sky"
encoded = encode(test_phrase)
decoded = decode(encoded)
print(f"Original: {repr(test_phrase)}")
print(f"Encoded:  {encoded}")
print(f"Decoded:  {repr(decoded)}")
print(f"Match:    {test_phrase == decoded}")

[104, 101, 108, 108, 111]
[272, 274, 288, 259, 115, 101]

Original: 'The night sky'
Encoded:  [84, 104, 256, 110, 105, 291, 264, 115, 107, 121]
Decoded:  'The night sky'
Match:    True


In [19]:
# Comprehensive round-trip validation

def validate(text, label=""):
    encoded = encode(text)
    decoded = decode(encoded)
    match = (text == decoded)
    status = "✓ PASS" if match else "✗ FAIL"
    print(f"  {status} [{label}]: {repr(text[:40])}...")
    return match

print("Validating tokenizer round-trip:")
print()

# Test with various text types
test_cases = [
    ("Pure ASCII", "The quick brown fox jumps over the lazy dog."),
    ("With accents", "The naïve résumé of François Müller"),
    ("Digits", "Pi is approximately 3.14159265358979323846"),
    ("Mixed", "Temperature: -40°C = -40°F exactly"),
    ("Symbols", "Math: ∀x ∈ ℝ, x² ≥ 0"),
    ("Repetition", "abcabcabcabcabcabcabcabc"),
    ("Full training text", training_text),
]

all_pass = True
for label, text in test_cases:
    result = validate(text, label)
    all_pass = all_pass and result

print()
print("All tests passed!" if all_pass else "Some tests FAILED — check your implementation.")

Validating tokenizer round-trip:

  ✓ PASS [Pure ASCII]: 'The quick brown fox jumps over the lazy '...
  ✓ PASS [With accents]: 'The naïve résumé of François Müller'...
  ✓ PASS [Digits]: 'Pi is approximately 3.141592653589793238'...
  ✓ PASS [Mixed]: 'Temperature: -40°C = -40°F exactly'...
  ✓ PASS [Symbols]: 'Math: ∀x ∈ ℝ, x² ≥ 0'...
  ✓ PASS [Repetition]: 'abcabcabcabcabcabcabcabc'...
  ✓ PASS [Full training text]: '\nThe night sky has fascinated humans for'...

All tests passed!


Real-world tokenizers don't run BPE on the entire text as a single stream. They first split the text using regex patterns, then run BPE on each chunk independently. This prevents undesirable merges across boundaries (e.g., merging the end of one word with the start of the next).

In [20]:
# GPT-2 style regex pre-tokenization
# Install: pip install regex
import regex as re

# This is the actual GPT-2 regex pattern (from openai's codebase)
# It splits on:
# - Contractions: 's, 't, 're, 've, 'm, 'll, 'd
# - Optional space + sequence of letters
# - Optional space + sequence of digits
# - Optional space + non-whitespace non-letter non-digit
# - Whitespace followed by non-whitespace (trailing whitespace)
# - Whitespace
gpt2_pattern = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

# Test on natural language
text1 = "Don't you think it's amazing? I'd say yes!"
chunks1 = re.findall(gpt2_pattern, text1)
print("Input:", repr(text1))
print("Chunks:", chunks1)
print()

# Test on code
text2 = """
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n - 1)
"""
chunks2 = re.findall(gpt2_pattern, text2)
print("Code chunks:", chunks2)

Input: "Don't you think it's amazing? I'd say yes!"
Chunks: ['Don', "'t", ' you', ' think', ' it', "'s", ' amazing', '?', ' I', "'d", ' say', ' yes', '!']

Code chunks: ['\n', 'def', ' factorial', '(', 'n', '):', '\n   ', ' if', ' n', ' ==', ' 0', ':', '\n       ', ' return', ' 1', '\n   ', ' return', ' n', ' *', ' factorial', '(', 'n', ' -', ' 1', ')', '\n']


#### Complexity:

- Naive training: O(num_merges x corpus_size)
- Naive encode: O(merges_applied x len(tokens))
- for optimized training: maintain a max-heap of pair counts + incremental updates 
- for optimized encode: same idea, track pair positions with a linked list

##### How 
Instead of recompute everything, then find the max maintain a running data structure that already knows the current counts, and only touch the small part that actually changed after each merge. 

Max-heap of pair counts — a heap lets you ask "what's the current highest-count pair?" in O(log n) instead of scanning every pair with max() each time (which is O(number of distinct pairs)).



#### how to improve this for prod further after using heap: 
- rust/c++ core with python 
- regex compilation caching, 
- support for add_special_tokens, padding, truncation as a higher-level API layer. 
- deterministic serialization format (vocab.json + merges.txt)